# PM Helper Agent

Run the setup cell once, then use the chat cell to process queued tasks.

In [ ]:
# Setup - Run once per session
import os
import json
import requests
from pathlib import Path
from datetime import datetime
from IPython.display import display, Markdown

API_URL = "https://api.anthropic.com/v1/messages"
ANTHROPIC_VERSION = "2023-06-01"
MODEL = "claude-haiku-4-20250514"

WORKING_DIR = Path.cwd()
AGENT_FILES = WORKING_DIR / "Agent Files"
HISTORY_DIR = WORKING_DIR / "History"
OUTPUT_DIR = WORKING_DIR / "Output"
PE_FRAMEWORK = WORKING_DIR.parent / "Claude PE Framework"
PM_QUEUE = PE_FRAMEWORK / "Agent Files" / "PM_Queue.md"

SYSTEM_PROMPT = (AGENT_FILES / "Instructions.md").read_text()
conversation_history = []

TOOLS = [
    {"name": "read_file", "description": "Read a file from Agent Files", "input_schema": {"type": "object", "properties": {"filename": {"type": "string"}}, "required": ["filename"]}},
    {"name": "write_file", "description": "Write a file to Agent Files", "input_schema": {"type": "object", "properties": {"filename": {"type": "string"}, "content": {"type": "string"}}, "required": ["filename", "content"]}},
    {"name": "list_files", "description": "List files in Agent Files", "input_schema": {"type": "object", "properties": {}}},
    {"name": "write_output", "description": "Write to Output directory", "input_schema": {"type": "object", "properties": {"filename": {"type": "string"}, "content": {"type": "string"}}, "required": ["filename", "content"]}},
    {"name": "read_pm_queue", "description": "Read the PM Task Queue", "input_schema": {"type": "object", "properties": {}}},
    {"name": "write_pm_queue", "description": "Write updated PM Task Queue", "input_schema": {"type": "object", "properties": {"content": {"type": "string"}}, "required": ["content"]}}
]

def execute_tool(name, inputs):
    if name == "read_file":
        path = AGENT_FILES / inputs["filename"]
        return path.read_text() if path.exists() else f"Not found: {inputs['filename']}"
    elif name == "write_file":
        path = AGENT_FILES / inputs["filename"]
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(inputs["content"])
        return f"Wrote {inputs['filename']}"
    elif name == "list_files":
        files = [f.name for f in AGENT_FILES.iterdir() if f.is_file()]
        return "\n".join(sorted(files)) if files else "No files"
    elif name == "write_output":
        OUTPUT_DIR.mkdir(exist_ok=True)
        path = OUTPUT_DIR / inputs["filename"]
        path.write_text(inputs["content"])
        return f"Wrote Output/{inputs['filename']}"
    elif name == "read_pm_queue":
        return PM_QUEUE.read_text() if PM_QUEUE.exists() else "Queue not found"
    elif name == "write_pm_queue":
        PM_QUEUE.write_text(inputs["content"])
        return "Updated PM_Queue.md"
    return f"Unknown tool: {name}"

def chat(message):
    global conversation_history
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        return "Error: ANTHROPIC_API_KEY not set"
    conversation_history.append({"role": "user", "content": message})
    while True:
        response = requests.post(API_URL, headers={"x-api-key": api_key, "anthropic-version": ANTHROPIC_VERSION, "content-type": "application/json"}, json={"model": MODEL, "max_tokens": 2048, "system": SYSTEM_PROMPT, "messages": conversation_history, "tools": TOOLS})
        if not response.ok:
            return f"API Error: {response.text}"
        data = response.json()
        conversation_history.append({"role": "assistant", "content": data["content"]})
        tool_uses = [b for b in data["content"] if b["type"] == "tool_use"]
        if not tool_uses:
            return "".join(b.get("text", "") for b in data["content"] if b["type"] == "text")
        tool_results = []
        for tool in tool_uses:
            print(f"[{tool['name']}]")
            result = execute_tool(tool["name"], tool["input"])
            tool_results.append({"type": "tool_result", "tool_use_id": tool["id"], "content": result})
        conversation_history.append({"role": "user", "content": tool_results})

def reset():
    global conversation_history
    conversation_history = []
    print("Reset.")

print(f"PM Helper ready. Model: {MODEL}")

In [ ]:
# Process Queue - Run to process all pending tasks
message = "Process all pending tasks in the queue."

response = chat(message)
display(Markdown(response))

In [ ]:
# View Queue
print(PM_QUEUE.read_text() if PM_QUEUE.exists() else "Queue not found")

In [ ]:
# Status Request
response = chat("Generate a full status report.")
display(Markdown(response))

In [ ]:
# Reset conversation
reset()